# 02 – Modélisation de la prévision de consommation

## Objectif du notebook

Ce notebook entraîne et compare plusieurs modèles de prévision de la consommation électrique totale journalière pour le programme Néovolt Grid+.

Le cas d’usage retenu est la prévision de consommation.

L’objectif métier est d’aider Néovolt à :
- anticiper les pics de demande ;
- préparer les achats d’énergie ;
- limiter les achats d’urgence ;
- améliorer la stabilité du réseau.

Les modèles testés sont :
- une baseline naïve ;
- un modèle linéaire Ridge ;
- un modèle Random Forest Regressor.

Les performances sont évaluées sur une période de test temporelle non vue par les modèles.

In [64]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import joblib

In [65]:
ROOT_DIR = Path.cwd()

if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parents[1]

OUTPUT_DIR = ROOT_DIR / "volet-c-data-scientist" / "outputs"
MODELS_DIR = ROOT_DIR / "volet-c-data-scientist" / "models"

MODELS_DIR.mkdir(parents=True, exist_ok=True)

DATASET_PATH = OUTPUT_DIR / "dataset_forecast_journalier.csv"

OUTPUT_DIR, MODELS_DIR, DATASET_PATH

(WindowsPath('c:/Users/Master/Desktop/Mohamed/1CPDA/Examen_S2/ExaS2/neovolt-grid-plus/volet-c-data-scientist/outputs'),
 WindowsPath('c:/Users/Master/Desktop/Mohamed/1CPDA/Examen_S2/ExaS2/neovolt-grid-plus/volet-c-data-scientist/models'),
 WindowsPath('c:/Users/Master/Desktop/Mohamed/1CPDA/Examen_S2/ExaS2/neovolt-grid-plus/volet-c-data-scientist/outputs/dataset_forecast_journalier.csv'))

In [66]:
# Chargement du dataset de prévision

df = pd.read_csv(DATASET_PATH)
df["date"] = pd.to_datetime(df["date"], errors="coerce")

print("Dimensions :", df.shape)
print("Date min :", df["date"].min())
print("Date max :", df["date"].max())

df.head()

Dimensions : (717, 24)
Date min : 2024-01-15 00:00:00
Date max : 2025-12-31 00:00:00


,date,consommation_totale_kwh,consommation_moyenne_kwh,temp_moyenne_c,temp_min_c,temp_max_c,degres_jour_chauffage,nb_releves,part_lignes_imputees_pct,annee,...,jour_annee,saison_automne,saison_ete,saison_hiver,saison_printemps,conso_lag_1,conso_lag_7,conso_rolling_7,conso_rolling_14,split
0,2024-01-15,21376.940,30.538486,3.407486,-0.576029,8.599843,13.592514,700,14.571429,2024,...,15,False,False,True,False,17120.185,21736.210,20245.401429,20061.844286,train
1,2024-01-16,21189.220,30.270314,3.733243,-0.348829,8.199386,13.266757,700,15.142857,2024,...,16,False,False,True,False,21376.940,21479.265,20194.077143,20077.648929,train
2,2024-01-17,21965.800,31.379714,1.345471,-1.302400,6.372986,15.654529,700,16.142857,2024,...,17,False,False,True,False,21189.220,21349.290,20152.642143,20076.225357,train
3,2024-01-18,21103.435,30.147764,4.807029,0.669086,9.361643,12.192971,700,14.285714,2024,...,18,False,False,True,False,21965.800,21407.125,20240.715000,20148.801786,train
4,2024-01-19,20888.260,29.840371,4.918629,1.870986,10.353814,12.081371,700,13.714286,2024,...,19,False,False,True,False,21103.435,21587.370,20197.330714,20118.216429,train


## Cible et logique d’évaluation

La variable cible est :

`consommation_totale_kwh`

Le modèle doit prédire la consommation totale journalière.

La séparation train/test respecte l’ordre temporel :
- `train` : période historique utilisée pour apprendre ;
- `test` : période future utilisée pour évaluer la capacité de prévision.

Cette approche évite de mélanger le passé et le futur, ce qui serait incorrect dans un problème de prévision temporelle.

In [67]:
# Séparation train / test

train_df = df[df["split"] == "train"].copy()
test_df = df[df["split"] == "test"].copy()

print("Train :", train_df.shape, train_df["date"].min(), "→", train_df["date"].max())
print("Test :", test_df.shape, test_df["date"].min(), "→", test_df["date"].max())

Train : (625, 24) 2024-01-15 00:00:00 → 2025-09-30 00:00:00
Test : (92, 24) 2025-10-01 00:00:00 → 2025-12-31 00:00:00


In [68]:
# Définition de la cible et des variables explicatives

target = "consommation_totale_kwh"

excluded_columns = [
    "date",
    "split",
    target
]

feature_columns = [col for col in df.columns if col not in excluded_columns]

feature_columns

['consommation_moyenne_kwh',
 'temp_moyenne_c',
 'temp_min_c',
 'temp_max_c',
 'degres_jour_chauffage',
 'nb_releves',
 'part_lignes_imputees_pct',
 'annee',
 'mois',
 'jour',
 'jour_semaine',
 'weekend',
 'jour_annee',
 'saison_automne',
 'saison_ete',
 'saison_hiver',
 'saison_printemps',
 'conso_lag_1',
 'conso_lag_7',
 'conso_rolling_7',
 'conso_rolling_14']

In [69]:
# Vérification des types des variables explicatives

df[feature_columns].dtypes

consommation_moyenne_kwh    float64
temp_moyenne_c              float64
temp_min_c                  float64
temp_max_c                  float64
degres_jour_chauffage       float64
nb_releves                    int64
part_lignes_imputees_pct    float64
annee                         int64
mois                          int64
jour                          int64
jour_semaine                  int64
weekend                       int64
jour_annee                    int64
saison_automne                 bool
saison_ete                     bool
saison_hiver                   bool
saison_printemps               bool
conso_lag_1                 float64
conso_lag_7                 float64
conso_rolling_7             float64
conso_rolling_14            float64
dtype: object

In [70]:
# Construction des matrices train/test

X_train = train_df[feature_columns]
y_train = train_df[target]

X_test = test_df[feature_columns]
y_test = test_df[target]

print("X_train :", X_train.shape)
print("X_test :", X_test.shape)

X_train : (625, 21)
X_test : (92, 21)


In [71]:
# Fonction d'évaluation des modèles

def mean_absolute_percentage_error(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def evaluate_model(model_name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    return {
        "modele": model_name,
        "MAE_kwh": round(mae, 2),
        "RMSE_kwh": round(rmse, 2),
        "MAPE_%": round(mape, 2)
    }

## Modèle 1 – Baseline naïve

La baseline naïve sert de point de comparaison.

Elle prédit que la consommation du jour sera égale à la consommation de la veille.

Cette baseline est importante : un modèle de machine learning doit faire mieux qu’une règle simple pour être réellement utile.

In [72]:
# Baseline naïve : prévision égale à la consommation de la veille

y_pred_baseline = test_df["conso_lag_1"]

baseline_results = evaluate_model(
    "Baseline naïve J-1",
    y_test,
    y_pred_baseline
)

baseline_results

{'modele': 'Baseline naïve J-1',
 'MAE_kwh': 1479.04,
 'RMSE_kwh': np.float64(2371.63),
 'MAPE_%': np.float64(8.84)}

## Modèle 2 – Régression Ridge

La régression Ridge est un modèle linéaire régularisé.

Elle sert de modèle simple, rapide et interprétable.  
Elle permet de vérifier si les variables calendaires, météo et historiques expliquent correctement la consommation.

In [73]:
# Modèle Ridge

ridge_model = Ridge(alpha=1.0)

ridge_model.fit(X_train, y_train)

y_pred_ridge = ridge_model.predict(X_test)

ridge_results = evaluate_model(
    "Ridge Regression",
    y_test,
    y_pred_ridge
)

ridge_results

{'modele': 'Ridge Regression',
 'MAE_kwh': 3.39,
 'RMSE_kwh': np.float64(4.17),
 'MAPE_%': np.float64(0.02)}

## Modèle 3 – Random Forest Regressor

Le Random Forest Regressor est un modèle non linéaire basé sur un ensemble d’arbres de décision.

Il peut capter des relations plus complexes entre :
- météo ;
- saison ;
- consommation passée ;
- calendrier ;
- tendance temporelle.

Il est plus puissant qu’un modèle linéaire, mais moins directement interprétable.

In [74]:
# Modèle Random Forest

rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

rf_results = evaluate_model(
    "Random Forest",
    y_test,
    y_pred_rf
)

rf_results

{'modele': 'Random Forest',
 'MAE_kwh': 26.23,
 'RMSE_kwh': np.float64(48.12),
 'MAPE_%': np.float64(0.17)}

In [75]:
# Comparaison des performances

results = pd.DataFrame([
    baseline_results,
    ridge_results,
    rf_results
])

results = results.sort_values(by="MAE_kwh")

results

,modele,MAE_kwh,RMSE_kwh,MAPE_%
1,Ridge Regression,3.39,4.17,0.02
2,Random Forest,26.23,48.12,0.17
0,Baseline naïve J-1,1479.04,2371.63,8.84


In [76]:
# Export des métriques

results.to_csv(OUTPUT_DIR / "model_forecast_metrics.csv", index=False, encoding="utf-8")

print("Métriques exportées :", OUTPUT_DIR / "model_forecast_metrics.csv")

Métriques exportées : c:\Users\Master\Desktop\Mohamed\1CPDA\Examen_S2\ExaS2\neovolt-grid-plus\volet-c-data-scientist\outputs\model_forecast_metrics.csv


In [77]:
fig = px.bar(
    results,
    x="modele",
    y="MAE_kwh",
    title="Comparaison des modèles – MAE",
    labels={
        "modele": "Modèle",
        "MAE_kwh": "MAE (kWh)"
    }
)

fig.show()

In [78]:
# Sélection du meilleur modèle selon la MAE

best_model_name = results.iloc[0]["modele"]

if best_model_name == "Baseline naïve J-1":
    best_predictions = y_pred_baseline
    best_model = None
elif best_model_name == "Ridge Regression":
    best_predictions = y_pred_ridge
    best_model = ridge_model
elif best_model_name == "Random Forest":
    best_predictions = y_pred_rf
    best_model = rf_model

print("Meilleur modèle :", best_model_name)

Meilleur modèle : Ridge Regression


In [79]:
# Dataset de prédictions sur la période de test

predictions_test = test_df[["date", target]].copy()
predictions_test["prediction_kwh"] = best_predictions
predictions_test["erreur_kwh"] = predictions_test[target] - predictions_test["prediction_kwh"]
predictions_test["erreur_absolue_kwh"] = predictions_test["erreur_kwh"].abs()
predictions_test["modele"] = best_model_name

predictions_test.head(15)

,date,consommation_totale_kwh,prediction_kwh,erreur_kwh,erreur_absolue_kwh,modele
625,2025-10-01,16368.420,16368.844831,-0.424831,0.424831,Ridge Regression
626,2025-10-02,16895.270,16885.198942,10.071058,10.071058,Ridge Regression
627,2025-10-03,17249.075,17243.453674,5.621326,5.621326,Ridge Regression
628,2025-10-04,12666.405,12664.165130,2.239870,2.239870,Ridge Regression
629,2025-10-05,12675.355,12670.568303,4.786697,4.786697,Ridge Regression
630,2025-10-06,17236.465,17231.396515,5.068485,5.068485,Ridge Regression
631,2025-10-07,16456.220,16455.927543,0.292457,0.292457,Ridge Regression
632,2025-10-08,17086.870,17078.043632,8.826368,8.826368,Ridge Regression
633,2025-10-09,16661.650,16661.451226,0.198774,0.198774,Ridge Regression
634,2025-10-10,16398.030,16402.885219,-4.855219,4.855219,Ridge Regression


In [80]:
# Visualisation réel vs prédit

predictions_long = predictions_test.melt(
    id_vars=["date"],
    value_vars=[target, "prediction_kwh"],
    var_name="serie",
    value_name="consommation_kwh"
)

fig = px.line(
    predictions_long,
    x="date",
    y="consommation_kwh",
    color="serie",
    markers=True,
    title=f"Consommation réelle vs prédite – {best_model_name}",
    labels={
        "date": "Date",
        "consommation_kwh": "Consommation (kWh)",
        "serie": "Série"
    }
)

fig.show()

In [81]:
# Statistiques sur les erreurs

predictions_test["erreur_absolue_kwh"].describe()

count    92.000000
mean      3.391749
std       2.440942
min       0.025878
25%       1.446475
50%       2.958427
75%       5.069190
max      10.071058
Name: erreur_absolue_kwh, dtype: float64

In [82]:
fig = px.bar(
    predictions_test,
    x="date",
    y="erreur_kwh",
    title=f"Erreur de prévision journalière – {best_model_name}",
    labels={
        "date": "Date",
        "erreur_kwh": "Erreur réelle - prédite (kWh)"
    }
)

fig.show()

In [83]:
# Importance des variables pour Random Forest

if best_model_name == "Random Forest":
    feature_importance = pd.DataFrame({
        "variable": feature_columns,
        "importance": rf_model.feature_importances_
    }).sort_values(by="importance", ascending=False)

    display(feature_importance.head(15))

    fig = px.bar(
        feature_importance.head(15),
        x="importance",
        y="variable",
        orientation="h",
        title="Top 15 des variables les plus importantes – Random Forest",
        labels={
            "importance": "Importance",
            "variable": "Variable"
        }
    )

    fig.show()
else:
    print("L'importance des variables est affichée uniquement si le meilleur modèle est Random Forest.")

L'importance des variables est affichée uniquement si le meilleur modèle est Random Forest.


In [84]:
# Sauvegarde du meilleur modèle si applicable

if best_model is not None:
    model_path = MODELS_DIR / "best_forecast_model.joblib"
    joblib.dump(best_model, model_path)
    print("Modèle sauvegardé :", model_path)
else:
    print("Le meilleur modèle est une baseline : aucun modèle scikit-learn à sauvegarder.")

Modèle sauvegardé : c:\Users\Master\Desktop\Mohamed\1CPDA\Examen_S2\ExaS2\neovolt-grid-plus\volet-c-data-scientist\models\best_forecast_model.joblib


In [85]:
# Export des prédictions de test

predictions_test.to_csv(
    OUTPUT_DIR / "forecast_predictions_test.csv",
    index=False,
    encoding="utf-8"
)

print("Prédictions exportées :", OUTPUT_DIR / "forecast_predictions_test.csv")

Prédictions exportées : c:\Users\Master\Desktop\Mohamed\1CPDA\Examen_S2\ExaS2\neovolt-grid-plus\volet-c-data-scientist\outputs\forecast_predictions_test.csv


## Conclusion de la modélisation

Ce notebook a permis de comparer plusieurs approches de prévision :
- une baseline naïve ;
- une régression Ridge ;
- un Random Forest Regressor.

Les modèles sont évalués sur une période de test temporelle non vue.

Les métriques utilisées sont :
- MAE ;
- RMSE ;
- MAPE.

Le meilleur modèle est sélectionné selon la MAE.  
Les prédictions de test et les métriques sont exportées pour l’évaluation finale.